In [1]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt

In [2]:
BATCH_SIZE = 32
IMAGE_SIZE = 640
CHANNELS=3
EPOCHS=15

In [3]:
input_shape = (640, 640, 3)

In [4]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    r'/home/smartcity_sp7/praveen/save_3_classes_data',
    shuffle=True,
    image_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

Found 954 files belonging to 3 classes.


In [5]:
class_names = dataset.class_names
class_names

['AK', 'BCC', 'SK']

In [6]:
def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.05, test_split=0.15, shuffle=True, shuffle_size=10):
    assert (train_split + test_split + val_split) == 1
    ds_size = len(ds)
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)
    return train_ds, val_ds, test_ds

In [7]:
train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)

In [8]:
len(train_ds)

24

In [9]:
resize_and_rescale = tf.keras.Sequential([
  layers.experimental.preprocessing.Resizing(IMAGE_SIZE, IMAGE_SIZE),
  layers.experimental.preprocessing.Rescaling(1./255),
])

In [10]:
input_shape = (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

model = models.Sequential([
    resize_and_rescale,
    layers.Conv2D(32, kernel_size = (3,3), activation='relu', input_shape=[640,640,3]),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64,  kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes, activation='softmax'),
])

model.build(input_shape=input_shape)

In [11]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [12]:
model.fit(
    train_ds,
    verbose=1,
    epochs=5,
)

Epoch 1/5
24/24 [==============================] - 291s 12s/step - loss: 1.2512 - accuracy: 0.3490
Epoch 2/5
24/24 [==============================] - 289s 12s/step - loss: 1.0957 - accuracy: 0.3737
Epoch 3/5
24/24 [==============================] - 268s 11s/step - loss: 1.0352 - accuracy: 0.4580
Epoch 4/5
24/24 [==============================] - 289s 12s/step - loss: 1.0213 - accuracy: 0.4922
Epoch 5/5
24/24 [==============================] - 309s 13s/step - loss: 0.9269 - accuracy: 0.5591


In [13]:
model.save_weights(r'/home/smartcity_sp7/praveen/W_E_I_G_H_T_S/cnn_weights_save')